# RimGraph-DG V4.5 — CPU-only decoded mask audit
This notebook does **not** train any model and does **not** require a GPU. It downloads/locates the glaucoma dataset, checks the actual decoded optic-disc/optic-cup supervision for ORIGA, REFUGE, and G1020, saves detailed audit CSVs to Google Drive, and exits before backbone/model construction.

In [ ]:
GLAUCOMMA_OVERRIDES = {
    'manual_data_dir': '',
    'run_name': 'paper_mask_audit_v45',
    'code_revision': 'rimgraph-dg-v4.5-cpu-mask-audit-20260809',
    'seeds': [2029],
    'fold_targets': ['ORIGA'],
    'run_global_baseline': False,
    'run_full_model': False,
    'run_optuna': False,
    'fast_dev_run': False,
    'num_workers': 0,
}

import hashlib, json, traceback, urllib.request
from pathlib import Path
import torch

print('=== V4.5 CPU MASK AUDIT LAUNCHER ===', flush=True)
print('PyTorch:', torch.__version__, flush=True)
print('CUDA available:', torch.cuda.is_available(), flush=True)
print('GPU is NOT required for this audit.', flush=True)
print('====================================', flush=True)

COMMIT = 'e03ce6ca910d49fa9dc94916752a3bd2c4ce0b29'
EXPECTED_RAW_SHA256 = '46ba27c7446662460456bc2bab186729c0df1b3e76533ce44f208150208335e2'
ROOT = f'https://raw.githubusercontent.com/AzizulHakim00/Glaucomma/{COMMIT}'
parts = [f'v4_parts/part_{i:02d}.py' for i in range(7)]
raw_code = '\n'.join(urllib.request.urlopen(f'{ROOT}/{name}', timeout=60).read().decode('utf-8') for name in parts)
actual_raw = hashlib.sha256(raw_code.encode('utf-8')).hexdigest()
assert actual_raw == EXPECTED_RAW_SHA256, f'Raw runner integrity check failed: {actual_raw}'

patch_specs = [
    ('runner_patch_v41.py', 'apply_v41'),
    ('runner_patch_v42.py', 'apply_v42'),
    ('runner_patch_v43.py', 'apply_v43'),
    ('runner_patch_v43_autograd.py', 'apply_v43_autograd'),
    ('runner_patch_v44_runtime.py', 'apply_v44_runtime'),
    ('runner_patch_v45_masks.py', 'apply_v45_masks'),
    ('runner_patch_v45_lowlabels.py', 'apply_v45_lowlabels'),
    ('runner_patch_v45_audit_only.py', 'apply_v45_audit_only'),
]
code = raw_code
for patch_name, function_name in patch_specs:
    print(f'[LAUNCHER] applying {patch_name}', flush=True)
    source = urllib.request.urlopen(f'{ROOT}/{patch_name}', timeout=60).read().decode('utf-8')
    namespace = {}
    exec(compile(source, patch_name, 'exec'), namespace, namespace)
    code = namespace[function_name](code)
compile(code, 'rimgraph_dg_v45_cpu_mask_audit.py', 'exec')
print('[LAUNCHER] CPU audit assembly PASSED', flush=True)

try:
    exec(code, globals(), globals())
    drive_root = Path('/content/drive/MyDrive/Glaucomma_RimGraphDG/paper_mask_audit_v45')
    marker = drive_root / 'MASK_AUDIT_COMPLETED.json'
    if not marker.exists():
        raise RuntimeError(f'Audit returned without completion marker: {marker}')
    print('\n✅ CPU MASK AUDIT VERIFIED COMPLETE', flush=True)
    print('Summary:', drive_root / 'mask_validity_by_source.csv', flush=True)
    print('Details:', drive_root / 'mask_validity_audit.csv', flush=True)
    print('Failures/missing cup:', drive_root / 'mask_decode_failures_or_missing_cup.csv', flush=True)
except BaseException:
    trace = traceback.format_exc()
    print('\n=== CPU MASK AUDIT FAILURE TRACEBACK ===', flush=True)
    print(trace, flush=True)
    try:
        failure_dir = Path('/content/drive/MyDrive/Glaucomma_RimGraphDG/paper_mask_audit_v45')
        failure_dir.mkdir(parents=True, exist_ok=True)
        (failure_dir / 'AUDIT_FAILURE_TRACEBACK.txt').write_text(trace, encoding='utf-8')
    except Exception:
        pass
    raise
